In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time

In [ ]:

MAX_TASKS = 2

In [ ]:

NPROC = 10
MEM = 20  

In [ ]:

os.makedirs('component_gas/energy/success', exist_ok=True)
os.makedirs('component_gas/energy/failure', exist_ok=True)

In [ ]:

opt_freq_path = "component_gas/opt+freq/success"
energy_path = "component_gas/energy"

In [ ]:
NEW_GAUSSIAN_KEYWORDS = "M062X/6-311+G(2d,p) em=gd3"

In [ ]:

def convert_out_to_gjf(out_file, gjf_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "gjf", "-O", gjf_file])

In [ ]:

def extract_charge_and_coordinates(gjf_file):
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index - 1
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index]
    coordinates_lines = lines[start_index + 1 : final_index]        
    
    return charge_and_multiplicity_line, coordinates_lines

In [ ]:

def modify_and_append_to_gjf(gjf_file, new_keywords, chk_directory, charge_and_coords):
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, gjf_file.replace('.gjf', '.chk'))
    with open(gjf_file, 'w') as file:
        file.write(f"%nprocshared={NPROC}\n")
        file.write(f"%mem={MEM}GB\n")
        file.write(f'%chk={chk_file}\n')
        file.write(f'# {new_keywords}\n\n')
        file.write(f'{gjf_file} energy ECW\n\n')
        file.write(charge_and_coords[0])  
        file.writelines(charge_and_coords[1])  
        file.write('\n\n')

In [ ]:

def run_gaussian_and_categorize(gjf_file, success_dir, failure_dir):
    base_name = os.path.basename(gjf_file).replace(".gjf", "")
    out_file = base_name + ".out"
    chk_file = base_name + ".chk"
    
    
    try:
        subprocess.run(["g16", gjf_file, out_file])

        
        with open(out_file, 'r') as file:
            contents = file.read()
            if "Normal termination" in contents:
                
                os.rename(gjf_file, os.path.join(success_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(success_dir, out_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return out_file, True
            else:
                
                os.rename(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(failure_dir, out_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(gjf_file, os.path.join(failure_dir, gjf_file))
        if os.path.exists(out_file):
            os.rename(out_file, os.path.join(failure_dir, out_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False    

In [ ]:
def check_and_move_files():
    
    
    for file in os.listdir(energy_path):
        if file.endswith(".out"):  
            file_path = os.path.join(energy_path, file)
            with open(file_path, 'r') as f:
                content = f.read()
                
                if "Normal termination" in content:
                    
                    target_dir = success_dir
                else:
                    
                    target_dir = failure_dir
                
                
                base_filename = os.path.splitext(file)[0]
                for ext in ['.gjf', '.chk', '.out']:
                    src_file = os.path.join(energy_path, base_filename + ext)
                    if os.path.exists(src_file):  
                        dst_file = os.path.join(target_dir, base_filename + ext)
                        os.rename(src_file, dst_file)

In [ ]:

success_dir = os.path.join(energy_path, "success")
failure_dir = os.path.join(energy_path, "failure")
os.makedirs(success_dir, exist_ok=True)
os.makedirs(failure_dir, exist_ok=True)

In [ ]:

def main():
    
    start_time = time.time() 
    
    
    for out_file in os.listdir(opt_freq_path):
        if out_file.endswith(".out"):
            
            source_file = os.path.join(opt_freq_path, out_file)
            target_file = os.path.join(energy_path, out_file.replace(".out", ".gjf"))

            
            convert_out_to_gjf(source_file, target_file)

            
            charge_and_coords = extract_charge_and_coordinates(target_file)

            
            modify_and_append_to_gjf(target_file, NEW_GAUSSIAN_KEYWORDS, energy_path, charge_and_coords)


    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        futures = [executor.submit(run_gaussian_and_categorize, os.path.join(energy_path, f), success_dir, failure_dir) for f in os.listdir(energy_path) if f.endswith(".gjf")]
        for future in as_completed(futures):
            pass  
    
    check_and_move_files()
    
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")


In [ ]:
if __name__ == "__main__":
    main()